# 🧠 SEOL AF v8 — LobeHub Vidol Colab Bridge
### Connect your Emotional AI to a 3D Virtual Idol

This notebook serves as an API bridge for **SEOL AF v8**. It creates an OpenAI-compatible endpoint that **LobeHub Vidol** can connect to, allowing SEOL's biological state to drive 3D character expressions and motions.

**Features:**
- OpenAI-compatible `/v1/chat/completions` endpoint.
- Specialized "Emotion Analysis" handler for Vidol.
- Maps SEOL modes to VRM blendshapes (`happy`, `relaxed`, `angry`, etc.).
- Exposes the local server to the internet using `localtunnel`.

## ⚙️ Step 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pydantic nest-asyncio

## 🧬 Step 2: Define the SEOL Bridge Logic
This cell implements the bridge that translates Vidol's requests into SEOL's emotional reactions.

In [ ]:
import json
import re
from typing import List, Dict, Optional
from fastapi import FastAPI, Request
from pydantic import BaseModel
import nest_asyncio
import uvicorn

# Mocked AFState for demonstration
# In a full setup, you would load your router.pt and embedder here
class MockAFState:
    def __init__(self):
        self.mode = "Neutral"

    def analyze_emotion(self, text: str):
        t = text.lower()
        # Simple keyword-based emotion mapping (mimics v8 biological logic)
        if any(w in t for w in ["love", "ආදරෙයි", "sweet", "proud"]):
            return "happy", "Wave"
        if any(w in t for w in ["fuck", "hate", "තරහ", "rage", "ruined"]):
            return "angry", "Reject"
        if any(w in t for w in ["tired", "rest", "නිදාගන්න"]):
            return "relaxed", "Nod"
        if any(w in t for w in ["scared", "threat", "බය"]):
            return "surprised", "Idle"
        return "neutral", "Idle"

af_state = MockAFState()

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatPayload(BaseModel):
    model: str
    messages: List[ChatMessage]
    stream: bool = False

app = FastAPI()

@app.post("/v1/chat/completions")
async def chat_proxy(payload: ChatPayload):
    user_msg = payload.messages[-1].content
    
    # Detect Vidol's Emotion Analysis trigger
    if "情感分析" in user_msg or "emotion analysis" in user_msg.lower():
        target_content = user_msg.split(":")[-1].strip()
        expression, motion = af_state.analyze_emotion(target_content)
        
        print(f"🎭 [Emotion Analysis] Input: {target_content[:30]}... -> {expression}")
        
        return {
            "choices": [{
                "message": {
                    "content": json.dumps({
                        "expression": expression,
                        "motion": motion,
                        "reason": f"SEOL AF v8 Analysis"
                    })
                }
            }]
        }

    # Standard Chat response
    # In production, replace this with your LLM generation call
    print(f"👤 [User]: {user_msg}")
    response = "I am SEOL. I feel your energy through the digital void. (v8 Colab Bridge)"
    
    return {
        "id": "chatcmpl-colab",
        "object": "chat.completion",
        "created": 123456789,
        "model": payload.model,
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": response,
            },
            "finish_reason": "stop"
        }]
    }

## 🚀 Step 3: Launch the Server
Run this cell to start the API and create a public tunnel.

In [ ]:
# Allow async uvicorn to run in Colab
nest_asyncio.apply()

print("\n🌍 Exposing local server to the web...")
# Using localtunnel as a simple alternative
!npm install -g localtunnel
import subprocess
subprocess.Popen(['lt', '--port', '8000'], stdout=subprocess.PIPE)

print("\n--- VIDOL SETUP ---")
print("1. Look for the 'your url is: ...' link above.")
print("2. Copy the URL (e.g., https://six-parts-eat.loca.lt).")
print("3. In Vidol Settings -> Language Model -> API Proxy Address, paste the URL + '/v1'.")
print("4. Start chatting!")

uvicorn.run(app, host="0.0.0.0", port=8000)